# Experiment configurations

## Objective and search domain

For $x \in [-4,4]^2$ and $c \ge 0$, minimise

$$
f_c(x) = \min\left\{
    1 + \lVert x-a \rVert_2^2,\;
    \lVert x-b \rVert_2^2
    + c\left[\max\left(0, \lVert x-b \rVert_2 - 0.5\right)\right]^2
\right\},
\qquad a = (-2,0),\quad b = (2,0).
$$

Basin A has a local minimum of 1 at $a$. Basin B has the global minimum
of 0 at $b$. Increasing $c$ shrinks the region where $f_c(x)<1$ and
steepens the B variant outside its center. The radius-0.5 center and the
objective inside it stay the same. The iteration changes the objective
values outside the center as well as the size of the discovery region.

`two_basin_function(x, c)` uses this geometry by default. Experiment 2B
passes optional `a`, `b`, and `r0` arguments to compare other geometries.


In [ ]:
import numpy as np

BASIN_A = np.array([-2.0, 0.0])
BASIN_B = np.array([2.0, 0.0])
CENTER_RADIUS = 0.5
DOMAIN_BOUNDS = np.array([[-4.0, 4.0], [-4.0, 4.0]])


def two_basin_function(x, c, a=BASIN_A, b=BASIN_B, r0=CENTER_RADIUS):
    """Evaluate the shared objective, optionally using another basin geometry."""
    distance_a = np.linalg.norm(np.asarray(x) - a)
    distance_b = np.linalg.norm(np.asarray(x) - b)
    return float(min(1 + distance_a**2,
                     distance_b**2 + c * max(0, distance_b - r0)**2))


def make_objective(c):
    return lambda x: two_basin_function(x, c)

## Experiment 1 settings

The next cell specifies the four methods, the analysis targets, and the run grid. Experiment 1 evaluates $c\in\{0,3,10,30\}$ with seeds 0-9 and 50 evaluations per method per seed. Four penalties, ten seeds, and four methods give **160 trials**.

`METHOD_PARAMS` contains parameters used by the main function. 

`ANALYSIS_SETTINGS` defines four strict targets:

| Target | Meaning |
|---|---|
| $f<1$ | Beat A's local minimum and discover B |
| $f<0.25$ | Enter B's fixed radius-0.5 center |
| $f<10^{-2}$ | Reach the first precision target |
| $f<10^{-4}$ | Reach the smaller precision target |

`RUN_SETTINGS` combines the penalty values, seeds, method names, budget, and problem geometry. The implementations and library defaults are explained in `methods.ipynb`.

In [ ]:
METHOD_PARAMS = {
    "BO": {"n_initial_points": 4, "acq_func": "EI", "noise": 1e-10, "xi": 0.0},
    "CMA-ES": {"sigma0": 0.25},
    "TuRBO-1": {"n_initial_points": 4},
    "Random": {},
}
ANALYSIS_SETTINGS = {
    "targets": [1.0, 0.25, 1e-2, 1e-4],
    "near_a_radius": 1.0,
    "late_evaluations": 20,
}
RUN_SETTINGS = {
    "c_values": [0.0, 3.0, 10.0, 30.0],
    "seeds": list(range(10)),
    "methods": list(METHOD_PARAMS),
    "budget": 50,
    "problem": {
        "a": BASIN_A.tolist(), "b": BASIN_B.tolist(),
        "r0": CENTER_RADIUS, "bounds": DOMAIN_BOUNDS.tolist(),
    },
}

## Experiment 2A settings

The next cell contains the settings specific to the paired BO continuation experiment. It uses penalties 10 and 30 and seeds **100-149**. Each initial trial runs ordinary EI for 12 calls. Only trials whose best value is still at least 1 continue into the four comparison variants.

- **`chosen_xi=0.5`**: the fixed positive EI offset used by the higher-xi variants during exploration. Ordinary EI uses zero.
- **`fork=12`**: the number of evaluated points before divergence
- **`explore_end=40`**: the number of cases to end divergence of cases.
- **`budget=50`**: budget given.
- **`probe_calls`** which evaluation to perform prove calls.
- **`discovery=1.0`** and **`precision=0.01`** define the strict objective targets used to evaluate the continuations.

In [ ]:
EXPERIMENT_2A_SETTINGS = {
    "penalties": [10, 30],
    "main_seeds": list(range(100, 150)),
    "chosen_xi": 0.5,
    "fork": 12,
    "explore_end": 40,
    "budget": 50,
    "probe_calls": list(range(13, 41, 3)),
    "discovery": 1.0,
    "precision": 0.01,
}


## Experiment 2B settings

CMA-ES uses penalties 3 and 30, seeds 1–20, and checkpoints at 50 and 200
evaluations. The original domain and minima come from the shared objective
settings. The dictionary below contains the factorial and sweep grids,
alternative geometries, physical initial spreads, update-rule variants,
and analysis cutoffs. These preserve the settings used by the existing plots.

The baseline comparison uses PyCMA's default population and its doubled size.
The parent-count sweep fixes the population at 12. Initial-spread tests reuse
the wide domain; update-rule tests return to the original shared domain.


In [ ]:
EXPERIMENT_2B_SETTINGS = {
    "penalties": [3, 30],
    "seeds": list(range(1, 21)),
    "budgets": [50, 200],
    "start_fractions": [0.05, 0.10, 0.15],
    "sigma0": 0.30,
    "factorial_populations": [6, 12],
    "factorial_parents": [3, 6],
    "population_sweep": [6, 8, 12, 16, 24, 32, 48],
    "parent_sweep": [2, 4, 6, 8, 10],
    "parent_sweep_population": 12,
    "wide_domain": {
        "lower": -20.0, "upper": 20.0,
        "a": [-10.0, 0.0], "b": [10.0, 0.0], "r0": CENTER_RADIUS,
    },
    "initial_spread": {
        "popsize": 12, "mu": 6,
        "initial_sds": [2.4, 12.0],
        "offsets": [0.2, 0.4, 0.6],
    },
    "intermediate_domain": {
        "lower": -10.0, "upper": 10.0,
        "a": [-5.0, 0.0], "b": [5.0, 0.0], "r0": CENTER_RADIUS,
        "initial_sd": 6.0,
    },
    "updates": {
        "popsize": 12, "mu": 6,
        "variants": {
            "Baseline": {"CMA_cmean": 1.0, "AdaptSigma": True},
            "Half mean step": {"CMA_cmean": 0.5, "AdaptSigma": True},
            "Fixed step size": {"CMA_cmean": 1.0, "AdaptSigma": False},
        },
    },
    "analysis": {
        "settling_cutoffs": [0.05, 0.025],  # Unit coordinates on the original domain.
        "settling_cutoff": 0.05,
        "near_cutoff": 0.40,  # Physical RMS distance for cross-domain comparisons.
    },
}


## Experiment 2C settings

The six TuRBO variants share penalties 10 and 30, seeds 100–149, and a
50-call budget. Only the initial trust-region length or failure tolerance
changes; the default appears once. The notebook resumes the existing
`results/experiment2c_sweep.jsonl` checkpoint and verifies matching settings.

The same section also controls the pilot size, checkpoint location, objective
targets, diagnostic neighbourhood, late-evaluation window, and confidence level.
Changing optimisation settings requires a separate checkpoint; the existing
600 trials remain available with the defaults below.


In [ ]:
EXPERIMENT_2C_SETTINGS = {
    "penalties": [10, 30],
    "main_seeds": list(range(100, 150)),
    "n_initial_points": 4,
    "budget": 50,
    "pilot_seed_count": 5,
    "checkpoint": "results/experiment2c_sweep.jsonl",
    "discovery": 1.0,
    "precision": 0.01,
    "near_a_radius": 1.0,
    "late_evaluations": 20,
    "confidence_level": 0.95,
    "variants": {
        "Default": {"length_init": 0.8, "failtol": 4},
        "Small region": {"length_init": 0.4, "failtol": 4},
        "Large region": {"length_init": 1.6, "failtol": 4},
        "Fast contraction": {"length_init": 0.8, "failtol": 2},
        "Slow contraction": {"length_init": 0.8, "failtol": 8},
        "Very slow contraction": {"length_init": 0.8, "failtol": 12},
    },
}


## Experiment 3 settings

Experiment 3 compares BO-EI and CMA-ES from two controlled initial designs at $c=30$. One design gives the optimiser a foothold near A and the other gives it a foothold near B. Each design uses 12 calls, leaving 38 calls for the optimiser.

The BO-EI comparison uses the same ten uniform-probe calls as Experiment 2A. The CMA-ES comparison uses the default initial step size from Experiment 1 and a smaller step size of $0.05$ in unit-box coordinates.

In [ ]:
EXPERIMENT_3_SETTINGS = {
    "penalty": 30,
    "seeds": list(range(300, 350)),
    "budget": EXPERIMENT_2A_SETTINGS["budget"],
    "probe_calls": EXPERIMENT_2A_SETTINGS["probe_calls"],
    "n_background": 8,
    "local_radii": [0.35, 0.39, 0.43, 0.47],
    "small_sigma": 0.05,
}